# 1000 Genomes + AlphaMissense Integration Pipeline

This notebook builds a per-variant annotation table by combining genotype data from the 1000 Genomes Project with predicted pathogenicity scores from AlphaMissense. Two outputs are produced.

**Pipeline steps:**

1. Download the **1000G manifest** which references all chromosome VCF files
2. For each chromosome: download 1000G VCF → convert to Parquet (SNVs only) → download gnomAD HGDP+1kG VCF → filter to **missense-confirmed SNVs** → delete raw files
3. Download **AlphaMissense hg38**
4. Convert AlphaMissense TSV → Parquet
5. Join 1000G × AlphaMissense — two outputs:
   - **Left join** (`1000G_alphamissense_left.parquet`): all missense SNVs; **43,443,258 rows**; `uniprot_id` null for variants not covered by AlphaMissense (large proteins)
   - **Inner join** (`1000G_alphamissense_inner.parquet`): only AM-scored variants; **32,063,253 rows**; `uniprot_id` always non-null after Step 6; used by the merge pipeline
6. **VEP enrichment**: query Ensembl VEP REST API for the ~75K unique variants missing `uniprot_id`; fill in SwissProt accessions; update both outputs

Both outputs share the same **12-column schema** (`TARGET_COLS_1000G`):

| Column | Type | Source | Description |
|--------|------|--------|-------------|
| `chrom` | String | VCF | Chromosome with `chr` prefix (e.g. `chr1`) |
| `pos` | Int64 | VCF | Genomic position (GRCh38) |
| `ref` | String | VCF | Reference allele (single nucleotide) |
| `alt` | String | VCF | Alternate allele (single nucleotide) |
| `sample_id` | String | VCF | Individual identifier (e.g. `HG00096`) |
| `dosage` | Int8 | VCF | Allele dosage: `1` = heterozygous, `2` = homozygous-alt |
| `uniprot_id` | String | AM / VEP | UniProt accession; null only in left join for variants with no SwissProt entry in either AM or VEP |
| `transcript_id` | String | AM | Ensembl transcript; null in left-join-only rows |
| `protein_variant` | String | AM | Amino acid substitution key (e.g. `G12D`); null in left-join-only rows |
| `am_pathogenicity` | Float64 | AM | Pathogenicity score 0–1; null in left-join-only rows |
| `am_class` | String | AM | `likely_benign` / `ambiguous` / `likely_pathogenic`; null in left-join-only rows |
| `af_1000g` | Float64 | VCF INFO | Alternate allele frequency across all 3,202 individuals |

> `gene_symbol` and `gene_name` are **not** added here — they are derived in the merge pipeline via a UniProt API lookup on `uniprot_id`.
>
> **Coverage note**: AlphaMissense does not cover all proteins. Variants in large proteins (>7,500 aa) such as DST/Dystonin are absent from the database. Step 6 fills in `uniprot_id` for these variants via Ensembl VEP (where a SwissProt entry exists); `am_pathogenicity` and `am_class` remain null for these variants.
>
> After Step 6 (VEP enrichment), `uniprot_id` null count in the left output decreases from 11,380,005 to a smaller irreducible set; the inner output is re-derived as the non-null subset.

## Prerequisites

```bash
pip install polars requests
```

All downloaded and generated files are stored in `C:\Users\daane\Desktop\DATA\1000G`.  
Adjust `DATA_DIR` before running on a different machine.

## Imports and Path Constants

All imports and file paths are defined here so every subsequent step can simply reference the constants. The data directory is created if it does not yet exist.

In [ ]:
import os
import glob
import gzip
import shutil
import urllib
import requests
import time

import polars as pl

DATA_DIR = r"C:\Users\daane\Desktop\DATA\1000G"
os.makedirs(DATA_DIR, exist_ok=True)

# 1000 Genomes
BASE_URL      = "https://ftp.1000genomes.ebi.ac.uk/vol1/ftp/data_collections/1000G_2504_high_coverage/working/20220422_3202_phased_SNV_INDEL_SV"
MANIFEST_URL  = f"{BASE_URL}/20220804_manifest.txt"
MANIFEST_PATH = os.path.join(DATA_DIR, "20220804_manifest.txt")

# AlphaMissense
AM_DOWNLOAD_LINK = "https://zenodo.org/records/10813168/files/AlphaMissense_hg38.tsv.gz?download=1"
AM_GZ_PATH       = os.path.join(DATA_DIR, "AlphaMissense_hg38.tsv.gz")
AM_TSV_PATH      = os.path.join(DATA_DIR, "AlphaMissense_hg38.tsv")
AM_PARQUET_PATH  = os.path.join(DATA_DIR, "alphamissense_hg38.parquet")

# gnomAD HGDP+1kG (VEP-annotated, used for missense filtering in Step 2)
GNOMAD_BASE_URL = "https://gnomad-public-us-east-1.s3.amazonaws.com/release/3.1.2/vcf/genomes"
GNOMAD_URL_FMT  = GNOMAD_BASE_URL + "/gnomad.genomes.v3.1.2.hgdp_tgp.{chrom}.vcf.bgz"

# Step 5 outputs — two AM-annotated outputs (left join = all missense; inner = AM-covered only)
G1000_LEFT_PATH  = os.path.join(DATA_DIR, "1000G_alphamissense_left.parquet")
G1000_INNER_PATH = os.path.join(DATA_DIR, "1000G_alphamissense_inner.parquet")

# Schema constants
COORD_KEY = ["chrom", "pos", "ref", "alt"]

# Per-chromosome output schema (Step 2 output); ac/an omitted (derivable from af, unused downstream)
VCF_SCHEMA = {
    "chrom": pl.Utf8, "pos": pl.Int64, "ref": pl.Utf8, "alt": pl.Utf8,
    "af": pl.Float64, "sample_id": pl.Utf8, "dosage": pl.Int8,
}

# Column order for both Step 5 outputs (12 cols; no gene_symbol/gene_name — derived in merge)
TARGET_COLS_1000G = [
    *COORD_KEY, "sample_id", "dosage",
    "uniprot_id", "transcript_id", "protein_variant",
    "am_pathogenicity", "am_class",
    "af_1000g",
]

# VCF conversion constants
GT_MAP = {"0|0": 0, "0|1": 1, "1|0": 1, "1|1": 2}
BATCH_SIZE_ROWS = 1_000_000   # flush to temp Parquet every 1 M rows

---
## Step 1 — Download Manifest

The manifest lists every file in the 2022 phased panel release with its MD5 checksum. We download it once and parse it to get the ordered list of per-chromosome VCF filenames.

**Output**: `20220804_manifest.txt` — 23 VCF files (chr1–chr22 + chrX)

In [ ]:
OVERRIDE = False

if os.path.exists(MANIFEST_PATH) and not OVERRIDE:
    print(f"Already exists: {MANIFEST_PATH}, skipping download.")
else:
    urllib.request.urlretrieve(MANIFEST_URL, MANIFEST_PATH)
    print(f"Saved manifest to {MANIFEST_PATH}")

# Keep only .vcf.gz entries (skip .tbi and any other files)
with open(MANIFEST_PATH) as f:
    vcf_files = [line.split()[0] for line in f if line.strip() and line.split()[0].endswith(".vcf.gz")]

print(f"\n{len(vcf_files)} VCF files found:")
for fname in vcf_files:
    print(f"  {fname}")

## Step 2 — Convert 1000G VCF → Parquet (SNV-only, missense-only)

### Step 2.1 — Conversion and Annotation Functions

The 2022 phased panel VCFs follow the standard VCF format:
- Lines starting with `#` are header/metadata.
- The `#CHROM` header line lists the 3,202 sample IDs in columns 9 onward.
- Each data row is one variant; columns 9 onward contain per-sample `GT` fields (e.g. `0|1`, `1|1`).

The INFO field encodes population-level statistics: `AC` (alternate allele count), `AN` (total allele count), `AF` (allele frequency). Multi-allelic sites carry comma-separated values; we take the first value throughout. We retain only `AF` (renamed to `af_1000g` in Step 5) — `AC` and `AN` are derivable from `AF` and are not used downstream.

We convert directly to one row per (variant, individual) pair where the individual carries the alternate allele, skipping the wide (3,202-column) intermediate entirely.

**SNV filter**: Only single-nucleotide variants (`len(ref) == 1` and `len(alt) == 1`) are retained. INDELs are discarded before any per-sample processing, so they are never written to disk.

**Genotype (GT) → dosage encoding:**

| GT string | Dosage | Meaning |
|-----------|--------|---------|
| `0\|0` | 0 | Homozygous reference → **excluded** (dosage = 0) |
| `0\|1` or `1\|0` | 1 | Heterozygous |
| `1\|1` | 2 | Homozygous alternate |
| anything else | −1 | Missing → **excluded** |

Only rows with dosage > 0 are kept, reducing output size dramatically (most individuals are ref/ref for any given variant).

**Batching strategy**: rows are accumulated in memory and flushed to a temporary Parquet file every `BATCH_SIZE_ROWS` (1 M rows). Batch files are merged with Polars' lazy API at the end. This keeps peak memory bounded regardless of chromosome size.

**`build_gnomad_missense_index`**: streams the gnomAD HGDP+1kG v3.1.2 VCF for the same chromosome and returns a `(chrom, pos_str, ref, alt) → bool` lookup. Only columns 0–7 (CHROM–INFO) are parsed via `split('\t', maxsplit=8)`, skipping the ~3,900 per-sample genotype columns. The VEP `Consequence` field is inspected; a variant is marked `True` if any transcript annotation contains `missense_variant`.

**Output per chromosome**: `chr{N}.parquet` — 7 columns (`chrom`, `pos`, `ref`, `alt`, `af`, `sample_id`, `dosage`); one row per (missense SNV, individual carrying the alt allele).

In [ ]:
def parse_info(s):
    """Parse VCF INFO string into a dict. E.g. 'AC=5;AN=6404;AF=0.001' -> {'AC': '5', ...}"""
    return dict(kv.split("=", 1) for kv in s.split(";") if "=" in kv)


def vcf_to_parquet(vcf_path, out_path):
    """Stream a chromosome VCF.gz to Parquet, then delete the raw VCF.

    One output row per (variant, individual) pair with dosage > 0.
    Only SNVs (len(ref) == 1 and len(alt) == 1) are retained; INDELs are discarded.
    Rows are accumulated in memory and flushed every BATCH_SIZE_ROWS to keep peak memory bounded.
    """
    chrom_name = os.path.basename(vcf_path).split(".")[1]
    tmp_prefix = out_path.replace(".parquet", "_tmp")

    print(f"Counting variants in {chrom_name} ...")
    with gzip.open(vcf_path, "rt") as f:
        n_variants = sum(1 for line in f if not line.startswith("#"))
    print(f"  {n_variants:,} variants")

    samples = []
    batch_rows = []
    batch_files = []

    with gzip.open(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("#CHROM"):
                # Sample IDs start at column 9 of the #CHROM header line
                samples = line.strip().split("\t")[9:]
                continue
            if line.startswith("#"):
                continue

            parts = line.rstrip("\n").split("\t")
            chrom_v, pos_str, _, ref, alt, _, _, info_str = parts[:8]

            # Skip INDELs — keep only single-nucleotide variants
            if len(ref) != 1 or len(alt) != 1:
                continue

            info = parse_info(info_str)

            # AF can be comma-separated for multi-allelic sites — take the first value
            af = float(info["AF"].split(",")[0]) if "AF" in info else None
            pos = int(pos_str)

            for i, gt_str in enumerate(parts[9:9 + len(samples)]):
                dosage = GT_MAP.get(gt_str, -1)
                if dosage <= 0:     # skip ref/ref (0) and missing (-1)
                    continue
                batch_rows.append((chrom_v, pos, ref, alt, af, samples[i], dosage))

            if len(batch_rows) >= BATCH_SIZE_ROWS:
                path = f"{tmp_prefix}_{len(batch_files)}.parquet"
                pl.DataFrame(batch_rows, schema=VCF_SCHEMA, orient="row").write_parquet(path)
                batch_files.append(path)
                print(f"  flushed batch {len(batch_files)}")
                batch_rows.clear()

    if batch_rows:
        path = f"{tmp_prefix}_{len(batch_files)}.parquet"
        pl.DataFrame(batch_rows, schema=VCF_SCHEMA, orient="row").write_parquet(path)
        batch_files.append(path)

    pl.scan_parquet(batch_files).sink_parquet(out_path)
    for p in batch_files:
        os.remove(p)

    os.remove(vcf_path)
    os.remove(vcf_path + ".tbi")
    print(f"  Saved {out_path} ({os.path.getsize(out_path) / 1e6:.1f} MB) | deleted raw VCF")


def build_gnomad_missense_index(vcf_path):
    """Stream gnomAD HGDP+1kG VCF; return (chrom, pos_str, ref, alt) -> is_missense bool.

    Only reads columns 0-7 (CHROM-INFO) via maxsplit=8 — skips the ~3,900 genotype columns.
    Returns True if any VEP transcript annotation contains 'missense_variant', False otherwise.
    Variants absent from gnomAD are absent from the returned dict (treated as null downstream).
    """
    vep_consequence_idx = None
    index = {}

    with gzip.open(vcf_path, "rt") as f:
        for line in f:
            if line.startswith("##INFO=<ID=vep"):
                if "Format:" in line:
                    fmt = line.split("Format:")[1].strip().strip('"').rstrip('">')
                    subfields = fmt.split("|")
                    if "Consequence" in subfields:
                        vep_consequence_idx = subfields.index("Consequence")
                continue
            if line.startswith("#"):
                continue

            fields = line.rstrip("\n").split("\t", maxsplit=8)
            chrom, pos, ref, alt = fields[0], fields[1], fields[3], fields[4]

            if len(ref) != 1 or len(alt) != 1:
                continue

            is_missense = False
            if vep_consequence_idx is not None:
                for entry in fields[7].split(";"):
                    if entry.startswith("vep="):
                        for transcript in entry[4:].split(","):
                            parts = transcript.split("|")
                            if len(parts) > vep_consequence_idx:
                                if "missense_variant" in parts[vep_consequence_idx]:
                                    is_missense = True
                                    break
                        break

            index[(chrom, pos, ref, alt)] = is_missense

    return index

### Step 2.2 — Per-Chromosome Download, Conversion, and Missense Filter

For each VCF in the manifest, three sub-steps run in order:

1. **1000G VCF → Parquet** (`chr{N}.parquet`): download `.vcf.gz` + `.tbi`, convert via `vcf_to_parquet` (INDELs discarded), delete raw VCF.
2. **gnomAD missense index** (`gnomad_{N}_index.parquet`): download `gnomad.genomes.v3.1.2.hgdp_tgp.{chrom}.vcf.bgz` from the gnomAD S3 bucket, stream through `build_gnomad_missense_index`, save as a Parquet lookup table.
3. **Missense filter**: join `chr{N}.parquet` against the index on `COORD_KEY`, keep only `is_missense == True` rows, write back.

**Restart safety — three levels of caching:**

| Condition | Behaviour |
|-----------|-----------|
| Both `chr{N}.parquet` **and** `gnomad_{N}_index.parquet` exist | Skip chromosome entirely |
| `chr{N}.parquet` exists but index does not | Skip 1000G download; proceed to gnomAD step |
| `gnomad_{N}_index.parquet` exists but 1000G parquet does not | Rare edge case; re-download 1000G, then load cached index |
| `gnomad_{N}.vcf.bgz` exists locally | Re-used as-is (not re-downloaded); never deleted by this pipeline |

Set `OVERRIDE = True` to force re-processing of all chromosomes regardless of existing files.

After all 23 chromosomes are processed: **43,443,258 missense SNV-sample pairs** across 3,202 individuals.

**Outputs per chromosome:**
- `chr{N}.parquet` — SNVs only, missense-confirmed, 7 columns
- `gnomad_{N}_index.parquet` — `(chrom, pos, ref, alt, is_missense)` lookup table, kept for reuse

In [ ]:
OVERRIDE = False

for vcf_name in vcf_files:
    chrom = vcf_name.split(".")[1]
    out_path          = os.path.join(DATA_DIR, f"{chrom}.parquet")
    gnomad_index_path = os.path.join(DATA_DIR, f"gnomad_{chrom}_index.parquet")
    gnomad_path       = os.path.join(DATA_DIR, f"gnomad_{chrom}.vcf.bgz")

    # Both outputs exist → chromosome fully processed; skip entirely
    if os.path.exists(out_path) and os.path.exists(gnomad_index_path) and not OVERRIDE:
        print(f"Already done: {chrom}, skipping.")
        continue

    print(f"\n{'='*50}")

    # ---- 1000G: download and convert (SNVs only) ----
    if os.path.exists(out_path) and not OVERRIDE:
        print(f"1000G parquet exists: {chrom}.parquet, skipping download.")
    else:
        vcf_path = os.path.join(DATA_DIR, vcf_name)
        print(f"Downloading {vcf_name} ...")
        urllib.request.urlretrieve(f"{BASE_URL}/{vcf_name}", vcf_path)
        urllib.request.urlretrieve(f"{BASE_URL}/{vcf_name}.tbi", vcf_path + ".tbi")
        print(f"  {os.path.getsize(vcf_path) / 1e6:.1f} MB")
        vcf_to_parquet(vcf_path, out_path)

    # ---- gnomAD: load cached index or build from VCF ----
    if os.path.exists(gnomad_index_path) and not OVERRIDE:
        print(f"Loading cached gnomAD index: gnomad_{chrom}_index.parquet")
        gnomad_df = pl.read_parquet(gnomad_index_path)
    else:
        if not os.path.exists(gnomad_path) or OVERRIDE:
            print(f"Downloading gnomAD {chrom} ...")
            urllib.request.urlretrieve(GNOMAD_URL_FMT.format(chrom=chrom), gnomad_path)
            print(f"  {os.path.getsize(gnomad_path) / 1e6:.1f} MB")
        else:
            print(f"Using cached gnomAD {chrom} ({os.path.getsize(gnomad_path) / 1e6:.1f} MB)")

        print(f"Building missense index ...", end=" ", flush=True)
        gnomad_index = build_gnomad_missense_index(gnomad_path)
        print(f"{len(gnomad_index):,} SNVs indexed")

        gnomad_df = (
            pl.DataFrame({
                "chrom":       [k[0] for k in gnomad_index],
                "pos":         [int(k[1]) for k in gnomad_index],
                "ref":         [k[2] for k in gnomad_index],
                "alt":         [k[3] for k in gnomad_index],
                "is_missense": list(gnomad_index.values()),
            })
            .with_columns(pl.col("pos").cast(pl.Int64))
        )
        gnomad_df.write_parquet(gnomad_index_path)
        print(f"Saved gnomad_{chrom}_index.parquet ({os.path.getsize(gnomad_index_path) / 1e6:.1f} MB)")
        # os.remove(gnomad_path)
        del gnomad_index

    # ---- Apply missense filter to chr*.parquet ----
    tmp_path = out_path + ".tmp"
    os.rename(out_path, tmp_path)
    n_before = pl.scan_parquet(tmp_path).select(pl.len()).collect().item()
    (
        pl.scan_parquet(tmp_path)
        .join(gnomad_df.lazy(), on=COORD_KEY, how="left")
        .filter(pl.col("is_missense"))
        .drop("is_missense")
        .sink_parquet(out_path)
    )
    n_after = pl.scan_parquet(out_path).select(pl.len()).collect().item()
    os.remove(tmp_path)
    print(f"{n_before:,} SNVs → {n_after:,} missense SNVs retained. Updated {chrom}.parquet.")
    del gnomad_df

In [ ]:
# ── Inspection: Step 2 ───────────────────────────────────────────────────────
_chrom_files = sorted(glob.glob(os.path.join(DATA_DIR, "chr*.parquet")))
print(f"Per-chromosome parquets found: {len(_chrom_files)} / 23")
print()
_total_rows = 0
for _p in _chrom_files:
    _n  = pl.scan_parquet(_p).select(pl.len()).collect().item()
    _mb = os.path.getsize(_p) / 1e6
    _total_rows += _n
    print(f"  {os.path.basename(_p):35s}  {_n:>10,} rows  {_mb:6.1f} MB")
print(f"\nTotal missense SNV-sample pairs: {_total_rows:,}")
del _chrom_files, _total_rows, _n, _mb, _p

---
## Step 3 — Download AlphaMissense hg38

**AlphaMissense** is a model developed by Google DeepMind that leverages AlphaFold protein structure predictions to score the pathogenicity of every possible single-nucleotide missense variant in the human proteome. Each variant receives:

- `am_pathogenicity`: a continuous score from **0** (benign) to **1** (pathogenic)
- `am_class`: a discrete label: `likely_benign`, `ambiguous`, or `likely_pathogenic`

We use the **hg38** build to match the GRCh38 reference genome used by TCGA-PAAD, 1000G, and gnomAD — enabling a unified join key across all three datasets.

**Coverage note**: AlphaMissense does not cover all proteins. Large proteins (>7,500 aa) such as DST/Dystonin are absent because AlphaFold2 cannot model them at full length. Approximately 194 genes have no AM scores; these variants will only appear in the left join output.

- **Source**: Zenodo, record 10813168
- **Reference**: Cheng et al. (2023). *Accurate proteome-wide missense variant effect prediction with AlphaMissense.* Science, 381(6664), eadg7492. https://doi.org/10.1126/science.adg7492

**Output**: `alphamissense_hg38.parquet` (the `.tsv.gz` is downloaded and decompressed temporarily; the TSV is deleted after conversion in Step 4)

In [ ]:
OVERRIDE = False

if os.path.exists(AM_PARQUET_PATH) and not OVERRIDE:
    print(f"Already exists: {AM_PARQUET_PATH}, skipping download.")
else:
    print(f"Downloading AlphaMissense to {AM_GZ_PATH} ...")
    urllib.request.urlretrieve(AM_DOWNLOAD_LINK, AM_GZ_PATH)
    print(f"  {os.path.getsize(AM_GZ_PATH) / 1e6:.1f} MB")

    print(f"Decompressing to {AM_TSV_PATH} ...")
    with gzip.open(AM_GZ_PATH, "rb") as f_in, open(AM_TSV_PATH, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

    os.remove(AM_GZ_PATH)
    print(f"Deleted {AM_GZ_PATH}")
    print("Done.")

---
## Step 4 — Convert AlphaMissense TSV → Parquet

The AlphaMissense TSV has no conventional header row — the file begins with `#`-prefixed comment lines followed by data rows without column names. We skip comments (`comment_prefix="#"`) and assign column names explicitly.

We drop the `genome` column (always `hg38`). The remaining columns propagated to the output are:

| Column | Description |
|---|---|
| `chrom` | Chromosome (e.g. `chr1`) |
| `pos` | GRCh38 genomic position |
| `ref` | Reference allele |
| `alt` | Alternate allele |
| `uniprot_id` | UniProt accession of the affected protein |
| `transcript_id` | Ensembl transcript identifier |
| `protein_variant` | Amino acid substitution (e.g. `G12D`) |
| `am_pathogenicity` | Pathogenicity score (0–1) |
| `am_class` | `likely_benign` / `ambiguous` / `likely_pathogenic` |

The conversion streams from disk to disk using Polars' lazy API so the full ~1.2 GB TSV is never loaded into memory.

**Input**: `AlphaMissense_hg38.tsv`  
**Output**: `alphamissense_hg38.parquet` — **71,697,556 rows**, 9 columns, 564 MB; no nulls in any column.

In [ ]:
OVERRIDE = False

if os.path.exists(AM_PARQUET_PATH) and not OVERRIDE:
    print(f"Already exists: {AM_PARQUET_PATH}, skipping.")
else:
    am_columns = [
        "chrom", "pos", "ref", "alt",
        "genome",
        "uniprot_id", "transcript_id", "protein_variant",
        "am_pathogenicity", "am_class",
    ]

    print(f"Streaming {AM_TSV_PATH} -> {AM_PARQUET_PATH} ...")
    (
        pl.scan_csv(
            AM_TSV_PATH,
            separator="\t",
            comment_prefix="#",
            has_header=False,
            new_columns=am_columns,
        )
        .drop("genome")
        .sink_parquet(AM_PARQUET_PATH)
    )

    os.remove(AM_TSV_PATH)
    print(f"Deleted {AM_TSV_PATH}")
    print("Done.")

---
## Step 5 — Join 1000G × AlphaMissense

Annotates the missense SNV table by joining each variant on `[chrom, pos, ref, alt]` against AlphaMissense hg38. Two outputs are written.

**Why two outputs?**

AlphaMissense does not score every protein — variants in very large proteins (>7,500 aa, e.g. DST/Dystonin, ~194 genes total) are absent from the database. An inner join silently drops them; a left join retains them with null AM columns. Both are useful for different downstream purposes:

| Output | Join type | `uniprot_id` | Use |
|--------|-----------|--------------|-----|
| `1000G_alphamissense_left.parquet` | Left | null for large-protein variants | Standalone 1000G analysis; preserves full missense coverage |
| `1000G_alphamissense_inner.parquet` | Inner (derived by filtering left) | Always non-null | Used by the merge pipeline |

**Deduplication**: AlphaMissense stores one row per (variant, transcript). Multi-transcript variants would fan out after the join. For each `(chrom, pos, ref, alt, sample_id)` key we keep the transcript with the **highest `am_pathogenicity`** (most pathogenic). Ties are broken by `transcript_id` ascending for determinism. This matches the TCGA pipeline strategy.

**Memory strategy**: per-chromosome `chr{N}.parquet` files are processed one at a time and written as `chr{N}_am.parquet` intermediates. The combined left output is assembled by Polars' lazy API from these intermediates; the inner output is derived by filtering the left output for non-null `uniprot_id`.

Both outputs have the same 12 columns (see title schema). **Left join**: 43,443,258 rows; 11,380,005 null `uniprot_id` (large-protein variants). **Inner join**: 32,063,253 rows; 0 nulls. Unique samples: 3,202. Unique variants: left 406,234 · inner 330,695. Dosage: 27.7M heterozygous · 15.8M homozygous-alt.

> `gene_symbol` and `gene_name` are **not** added here — they are derived in the merge pipeline via a UniProt API lookup on `uniprot_id`.

In [ ]:
OVERRIDE = False

if os.path.exists(G1000_LEFT_PATH) and os.path.exists(G1000_INNER_PATH) and not OVERRIDE:
    print("Both outputs exist, skipping.")
    print(f"  Left:  {G1000_LEFT_PATH}")
    print(f"  Inner: {G1000_INNER_PATH}")
else:
    chrom_files = sorted(glob.glob(os.path.join(DATA_DIR, "chr*.parquet")))
    print(f"Found {len(chrom_files)} chromosome parquet(s).")

    am = (
        pl.scan_parquet(AM_PARQUET_PATH)
        .with_columns(pl.col("pos").cast(pl.Int64))
    )

    chrom_result_files = []
    for chrom_path in chrom_files:
        chrom = os.path.basename(chrom_path).replace(".parquet", "")
        result_path = os.path.join(DATA_DIR, f"{chrom}_am.parquet")

        if os.path.exists(result_path) and not OVERRIDE:
            print(f"Already exists: {chrom}_am.parquet, skipping.")
        else:
            print(f"Processing {chrom} ...", end=" ", flush=True)
            df = (
                pl.scan_parquet(chrom_path)
                .join(am, on=COORD_KEY, how="left")
                .collect()
                .sort(["am_pathogenicity", "transcript_id"], descending=[True, False], nulls_last=True)
                .unique(subset=[*COORD_KEY, "sample_id"], keep="first", maintain_order=True)
                .rename({"af": "af_1000g"})
                .select(TARGET_COLS_1000G)
            )
            df.write_parquet(result_path)
            print(f"{df.height:,} rows ({os.path.getsize(result_path) / 1e6:.1f} MB)")
            del df
        chrom_result_files.append(result_path)

    # Left join output — all missense SNVs; uniprot_id null for large-protein variants
    # Handle cached chr_am.parquet files created before af→af_1000g rename
    print(f"\nCombining {len(chrom_result_files)} files -> left output ...")
    _combined = pl.scan_parquet(chrom_result_files)
    if "af" in _combined.schema:
        _combined = _combined.rename({"af": "af_1000g"})
    _combined.select(TARGET_COLS_1000G).sink_parquet(G1000_LEFT_PATH)
    print(f"  Left:  {G1000_LEFT_PATH}  ({os.path.getsize(G1000_LEFT_PATH) / 1e6:.1f} MB)")

    # Inner output — AM-covered variants only; uniprot_id always non-null
    (
        pl.scan_parquet(G1000_LEFT_PATH)
        .filter(pl.col("uniprot_id").is_not_null())
        .sink_parquet(G1000_INNER_PATH)
    )
    print(f"  Inner: {G1000_INNER_PATH}  ({os.path.getsize(G1000_INNER_PATH) / 1e6:.1f} MB)")
    print("Done.")

In [ ]:
_left = pl.read_parquet(G1000_LEFT_PATH)
key_cols = [*COORD_KEY, "sample_id"]

dupes = (
    _left
    .group_by(key_cols)
    .len()
    .filter(pl.col("len") > 1)
)

if len(dupes) == 0:
    print("No duplicate keys — every (chrom, pos, ref, alt, sample_id) is unique.")
else:
    print(f"WARNING: {len(dupes):,} duplicate key groups ({dupes['len'].sum():,} rows affected).\n")

    dupes_full = _left.join(dupes.select(key_cols), on=key_cols, how="inner")

    non_key_cols = [c for c in _left.columns if c not in key_cols]
    print("Variation within duplicate key groups (per column):")
    for col in non_key_cols:
        n_varying = (
            dupes_full
            .group_by(key_cols)
            .agg(pl.col(col).n_unique().alias("n_unique"))
            .filter(pl.col("n_unique") > 1)
            .height
        )
        label = f"varies in {n_varying:,} / {len(dupes):,} groups" if n_varying > 0 else "constant in all groups"
        print(f"  {col:30s}  {label}")

    print(f"\nTop duplicate groups (most copies):")
    print(dupes.sort("len", descending=True).head(10))

del _left

---
## Step 6 — VEP Enrichment: Fill `uniprot_id` for Large-Protein Variants

AlphaMissense does not score variants in proteins longer than ~7,500 amino acids (~194 genes including DST, AHNAK, TTN). These variants are retained in the left join output with null `uniprot_id`. This step fills in those gaps using the **Ensembl VEP REST API**.

Only the ~75K unique `(chrom, pos, ref, alt)` combinations with null `uniprot_id` are queried — not the full row count. For each variant the UniProt accession is extracted using a three-level priority:

1. **Canonical transcript + SwissProt** — the MANE Select transcript with a manually reviewed UniProt entry; most reliable.
2. **Any transcript + SwissProt** — same reviewed quality; recovers large proteins like TTN whose SwissProt entry (Q8WZ42) is attached to a non-canonical isoform rather than the MANE Select transcript.
3. **Canonical transcript + TrEMBL** — unreviewed, but the canonical (MANE Select) transcript anchors the gene identity reliably. Used only when no SwissProt entry exists for the protein at all.

Levels 1–2 prefer SwissProt regardless of isoform because SwissProt maintains one canonical entry per protein — every transcript of TTN maps to the same Q8WZ42. Level 3 accepts an unreviewed entry only when the canonical transcript makes the gene identity unambiguous.

Results are cached to `vep_uniprot_cache.parquet`. The apply cell (below) joins the cache against the left parquet on `COORD_KEY` and coalesces into `uniprot_id`; the inner parquet is then re-derived as the non-null subset. Both updated files are written atomically (tmp → rename).

> **Irreducible nulls**: A small fraction of variants remain null after this step:
> - Proteins with no UniProt entry of any kind.
> - Variants at pseudogene or lncRNA loci where gnomAD's older Ensembl annotation called `missense_variant` but current VEP returns only non-coding consequences. These are annotation-version artefacts and cannot be resolved.

**Outputs**: `vep_uniprot_cache.parquet` (cache), updated `1000G_alphamissense_left.parquet`, updated `1000G_alphamissense_inner.parquet`

In [ ]:
import math
from tqdm.auto import tqdm

VEP_CACHE_PATH = os.path.join(DATA_DIR, "vep_uniprot_cache.parquet")
VEP_URL     = "https://rest.ensembl.org/vep/human/region"
VEP_HEADERS = {"Content-Type": "application/json", "Accept": "application/json"}
VEP_BATCH   = 200
MAX_BATCHES = None   # set to an int (e.g. 5) for testing; None = run all batches
OVERRIDE    = False

if os.path.exists(VEP_CACHE_PATH) and not OVERRIDE:
    print(f"Cache exists: {VEP_CACHE_PATH}  ({os.path.getsize(VEP_CACHE_PATH) / 1e3:.1f} KB)")
else:
    null_variants = (
        pl.scan_parquet(G1000_LEFT_PATH)
        .filter(pl.col("uniprot_id").is_null())
        .select(COORD_KEY)
        .unique()
        .collect()
    )
    print(f"Unique variants to query: {null_variants.height:,}")

    # Ensembl uses no 'chr' prefix: chr1 → 1, chrX → X
    vep_inputs = [
        f"{row['chrom'].replace('chr', '')} {row['pos']} . {row['ref']} {row['alt']} . . ."
        for row in null_variants.iter_rows(named=True)
    ]

    batch_offsets = list(range(0, len(vep_inputs), VEP_BATCH))
    if MAX_BATCHES is not None:
        batch_offsets = batch_offsets[:MAX_BATCHES]

    results = {}
    for i in tqdm(batch_offsets, desc="VEP batches", unit="batch"):
        batch = vep_inputs[i : i + VEP_BATCH]

        for attempt in range(5):
            resp = requests.post(
                VEP_URL,
                headers=VEP_HEADERS,
                json={"variants": batch, "uniprot": 1, "canonical": 1},
            )
            if resp.status_code == 429:
                wait = int(resp.headers.get("Retry-After", 30))
                tqdm.write(f"  Rate-limited — sleeping {wait}s ...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            break

        for variant in resp.json():
            inp = variant.get("input", "")
            tcs = variant.get("transcript_consequences", [])
            uniprot = None

            # 1. Canonical + SwissProt; 2. Any + SwissProt (e.g. TTN Q8WZ42 on non-canonical);
            # 3. Canonical + TrEMBL (canonical transcript anchors gene identity).
            for tc in tcs:
                if tc.get("canonical") == 1 and tc.get("swissprot"):
                    uniprot = tc["swissprot"][0].split(".")[0]
                    break
            if uniprot is None:
                for tc in tcs:
                    if tc.get("swissprot"):
                        uniprot = tc["swissprot"][0].split(".")[0]
                        break
            if uniprot is None:
                for tc in tcs:
                    if tc.get("canonical") == 1 and tc.get("trembl"):
                        uniprot = tc["trembl"][0].split(".")[0]
                        break

            results[inp] = uniprot

        time.sleep(0.07)   # stay under 15 req/s

    vep_cache = pl.DataFrame({
        "chrom":          [f"chr{v.split()[0]}" for v in results],
        "pos":            [int(v.split()[1])     for v in results],
        "ref":            [v.split()[3]          for v in results],
        "alt":            [v.split()[4]          for v in results],
        "uniprot_id_vep": list(results.values()),
    }).with_columns(pl.col("pos").cast(pl.Int64))

    filled = vep_cache["uniprot_id_vep"].is_not_null().sum()
    print(f"Variants queried: {vep_cache.height:,}  (of {null_variants.height:,} total)")
    print(f"UniProt IDs found: {filled:,} / {vep_cache.height:,}  ({100 * filled / vep_cache.height:.1f}%)")

    vep_cache.write_parquet(VEP_CACHE_PATH)
    print(f"Cache saved: {VEP_CACHE_PATH}  ({os.path.getsize(VEP_CACHE_PATH) / 1e3:.1f} KB)")

In [ ]:
# Apply VEP cache: coalesce uniprot_id in left parquet; re-derive inner
if not os.path.exists(VEP_CACHE_PATH):
    print("No cache found — run the VEP query cell above first.")
else:
    vep_cache = pl.read_parquet(VEP_CACHE_PATH)

    _null_before = (
        pl.scan_parquet(G1000_LEFT_PATH)
        .filter(pl.col("uniprot_id").is_null())
        .select(pl.len())
        .collect()
        .item()
    )

    _tmp = G1000_LEFT_PATH + ".tmp"
    (
        pl.scan_parquet(G1000_LEFT_PATH)
        .join(vep_cache.lazy(), on=COORD_KEY, how="left")
        .with_columns(pl.coalesce(["uniprot_id", "uniprot_id_vep"]).alias("uniprot_id"))
        .drop("uniprot_id_vep")
        .sink_parquet(_tmp)
    )
    os.replace(_tmp, G1000_LEFT_PATH)

    (
        pl.scan_parquet(G1000_LEFT_PATH)
        .filter(pl.col("uniprot_id").is_not_null())
        .sink_parquet(G1000_INNER_PATH)
    )

    _null_after = (
        pl.scan_parquet(G1000_LEFT_PATH)
        .filter(pl.col("uniprot_id").is_null())
        .select(pl.len())
        .collect()
        .item()
    )
    print(f"uniprot_id nulls — before: {_null_before:,}  after: {_null_after:,}  filled: {_null_before - _null_after:,}")
    print(f"Left:  {G1000_LEFT_PATH}  ({os.path.getsize(G1000_LEFT_PATH) / 1e6:.1f} MB)")
    print(f"Inner: {G1000_INNER_PATH}  ({os.path.getsize(G1000_INNER_PATH) / 1e6:.1f} MB)")

---
## Summary

| Step | Input | Output | Notes |
|------|-------|--------|-------|
| 1 — Download manifest | FTP URL | `20220804_manifest.txt` | Lists all 23 chromosome VCFs; skips if exists |
| 2.1 — Define functions | — | — | `parse_info`, `vcf_to_parquet` (SNVs only, `af` only), `build_gnomad_missense_index` |
| 2.2 — Per-chromosome loop | Manifest | `chr{N}.parquet` × 23 (7 cols, 43,443,258 total rows) | Download 1000G VCF → convert (SNVs only) → download gnomAD v3.1.2 VCF → filter to missense-confirmed SNVs → delete both raw VCFs; skips if exists |
| 3 — Download AlphaMissense | Zenodo URL | `alphamissense_hg38.parquet` | ~1.2 GB compressed; TSV decompressed then deleted after conversion; skips if Parquet exists |
| 4 — Convert AlphaMissense | `AlphaMissense_hg38.tsv` | `alphamissense_hg38.parquet` (71,697,556 rows, 9 cols, 564 MB) | Streamed via lazy API; drops `genome` column |
| 5 — Join (left) | `chr*.parquet` + `alphamissense_hg38.parquet` | `1000G_alphamissense_left.parquet` (12 cols, 43,443,258 rows) | Left join; deduplication by max `am_pathogenicity`; `af` renamed to `af_1000g`; 11,380,005 rows have null `uniprot_id` |
| 5 — Join (inner) | `1000G_alphamissense_left.parquet` | `1000G_alphamissense_inner.parquet` (12 cols, 32,063,253 rows) | Filter left for non-null `uniprot_id`; used by merge pipeline |
| 6 — VEP query | `1000G_alphamissense_left.parquet` | `vep_uniprot_cache.parquet` | Batch-query Ensembl VEP for ~75K null-`uniprot_id` variants; 3-level priority (canonical+SwissProt → any+SwissProt → canonical+TrEMBL); skips if cache exists |
| 6 — VEP apply | `vep_uniprot_cache.parquet` + left parquet | updated left + inner | Coalesce VEP accessions into left; re-derive inner; always runs if cache exists |